In [ ]:
# Sprawdzenie i instalacja brakujących pakietów
required_packages <- c(
  "tidyverse",   # Do manipulacji danymi
  "broom",       # Tidyverse dla modeli statystycznych
  "car",         # Narzędzia do analizy regresji
  "performance", # Ocena modeli statystycznych
  "corrplot",    # Wizualizacja macierzy korelacji
  "lmtest",      # Testy dla modeli liniowych
  "olsrr",       # Narzędzia do regresji liniowej
  "caret",       # Narzędzia do uczenia maszynowego
  "recipes"      # Przetwarzanie wstępne danych
)

# Instalacja brakujących pakietów
new_packages <- required_packages[!(required_packages %in% installed.packages()[,"Package"])]
if(length(new_packages)) install.packages(new_packages)

# Ładowanie bibliotek
library(tidyverse)
library(broom)
library(car)
library(performance)
library(corrplot)
library(lmtest)
library(olsrr)
library(caret)
library(recipes)

# Ustawienie ziarna losowości
seed <- 1333
set.seed(seed)

Funkcje pomocnicze 

In [ ]:
# Funkcja do obliczania VIF
vifs_calculation <- function(df, entry_name = NULL) {
  # Obliczanie VIF
  vif_results <- car::vif(lm(Pierścienie ~ ., data = cbind(df, Pierścienie = rnorm(nrow(df)))))
  
  # Tworzenie ramki danych wynikowej
  if (!is.null(entry_name)) {
    vif_df <- data.frame(NAME_INDEX = entry_name, t(vif_results))
    rownames(vif_df) <- NULL
    vif_df <- vif_df %>% column_to_rownames("NAME_INDEX")
  } else {
    vif_df <- data.frame(t(vif_results))
  }
  
  return(vif_df)
}

# Funkcja F-test z permutacjami
F_test_permutation <- function(df, y, test_cols, n_perm) {
  results <- list()
  X_reduced <- df %>% select(-all_of(test_cols))
  
  for (test_col in test_cols) {
    full_features <- test_cols[test_cols != test_col]
    X_full <- df %>% select(-all_of(full_features))
    
    # Budowanie modeli
    model_reduced <- lm(y ~ ., data = cbind(X_reduced, y = y))
    model_full <- lm(y ~ ., data = cbind(X_full, y = y))
    
    # Obliczanie statystyki F
    anova_res <- anova(model_reduced, model_full)
    f_stat <- anova_res$F[2]
    
    # Procedura permutacji
    f_stats <- numeric(n_perm)
    for (i in 1:n_perm) {
      X_perm <- X_full
      X_perm[[test_col]] <- sample(X_full[[test_col]])
      model_perm <- lm(y ~ ., data = cbind(X_perm, y = y))
      f_stats[i] <- anova(model_reduced, model_perm)$F[2]
    }
    
    # Obliczanie p-value
    f_pos_sum <- sum(f_stats >= f_stat)
    p_value <- (f_pos_sum + 1) / (n_perm + 1)
    
    # Dodawanie wyników
    results[[test_col]] <- data.frame(
      `Tested feature` = test_col,
      `F statistic` = f_stat,
      `P_statistic` = f_pos_sum,
      `P value` = p_value,
      check.names = FALSE
    )
  }
  
  return(bind_rows(results) %>% column_to_rownames("Tested feature"))
}

Ładowanie danych

In [ ]:
# WCZYTANIE DANYCH
COL_NAMES <- c('Płeć', 'Długość', 'Średnica', 'Wysokość', 'Cała_waga', 'Waga_po_obraniu', 'Waga_trzewi', 'Waga_powłoki', 'Pierścienie')

# Sprawdzenie czy plik istnieje i wczytanie danych
if (!file.exists('abalone_data_set.csv')) {
  # Jeśli plik nie istnieje, można użyć odpowiednika fetch_ucirepo w R
  # Tutaj zakładam, że mamy plik CSV
  stop("Plik danych nie istnieje. Proszę zapewnić plik 'abalone_data_set.csv'")
} else {
  df <- read.csv('abalone_data_set.csv', col.names = COL_NAMES)
}

# PRZYGOTOWANIE DANYCH
X_reduced <- df %>% select(-Pierścienie)
y <- df %>% select(Pierścienie)

# Kodowanie zmiennych kategorycznych
X_reduced <- X_reduced %>% 
  mutate(
    Płeć_M = as.numeric(Płeć == "M"),
    Płeć_F = as.numeric(Płeć == "F"),
    Płeć_I = as.numeric(Płeć == "I")
  ) %>% 
  select(Płeć_I, Płeć_M, Płeć_F, everything(), -Płeć)

In [ ]:
# ANALIZA KORELACJI
sex_columns <- c('Płeć_I', 'Płeć_M', 'Płeć_F')

for (sex in sex_columns) {
  cor_data <- X_reduced %>% 
    filter(!!sym(sex) == 1) %>% 
    select(-all_of(sex_columns)) %>% 
    cor()
  
  corrplot(cor_data, method = "number", type = "upper", tl.cex = 0.8)
  title(paste("Korelacja dla", sex))
}

Analiza modelu dla konkretnej płci 

In [ ]:
# DOBÓR ZMIENNYCH OBJAŚNIAJĄCYCH
SEX <- 'I'
filt <- X_reduced[[paste0('Płeć_', SEX)]] == 1

y_local <- y[filt, , drop = FALSE]
X_old <- X_reduced[filt, ] %>% select(-all_of(sex_columns))

# Tworzenie nowych cech
X_new <- X_old %>% 
  mutate(
    `Długość/Wysokość` = Długość / Wysokość,
    `Masa/objętość` = Cała_waga / (Długość * Średnica * Wysokość),
    `Waga_powłoki/Cała_waga` = Waga_powłoki / Cała_waga,
    `Waga_po_obraniu/Cała_waga` = Waga_po_obraniu / Cała_waga,
    `Waga_trzewi/Cała_waga` = Waga_trzewi / Cała_waga,
    `Waga_powłoki/Waga_po_obraniu` = Waga_powłoki / Waga_po_obraniu,
    `Waga_trzewi/Waga_po_obraniu` = Waga_trzewi / Waga_po_obraniu,
    `Waga_trzewi/Waga_powłoki` = Waga_trzewi / Waga_powłoki
  )

# Usuwanie wierszy z brakującymi lub nieskończonymi wartościami
invalid_rows <- X_new %>% 
  mutate_all(list(~ is.na(.) | is.infinite(.))) %>% 
  apply(1, any)

X_old <- X_old[!invalid_rows, ]
X_new <- X_new[!invalid_rows, ]
y_local <- y_local[!invalid_rows, , drop = FALSE]

NEW_FEATURE_NAMES <- setdiff(colnames(X_new), colnames(X_old))

Normalność modelu

In [ ]:
# Model liniowy ze stałą
model_old <- lm(Pierścienie ~ ., data = cbind(X_old, Pierścienie = y_local$Pierścienie))

# Reszty
residuals <- residuals(model_old)

# Wykresy diagnostyczne
par(mfrow = c(1, 2))
# Wykres reszt vs przewidywane wartości
plot(fitted(model_old), residuals, xlab = "Wartości przewidywane", ylab = "Reszty")
abline(h = 0, col = "red")
title("Wykres reszt vs. przewidywane wartości")

# Wykres QQ
qqnorm(residuals)
qqline(residuals, col = "red")
title("Wykres QQ reszt modelu")

# Testy diagnostyczne
# Test normalności Shapiro-Wilka
normality_test <- shapiro.test(residuals)
cat("P value testu Shapiro-Wilka:", normality_test$p.value, "\n")

# Test heteroskedastyczności Breusch-Pagan
homoskedacity_test <- bptest(model_old)
cat("Breusch-Pagan test p-value:", homoskedacity_test$p.value, "\n")

Redukcja nowych zmiennych 

In [ ]:
# VIF dla modelu podstawowego
vifs_basic <- vifs_calculation(X_old, "Model bazowy")
print(vifs_basic)

# VIF po redukcji
reduce_basic <- c('Cała_waga', 'Długość')
vifs_basic_after_red <- vifs_calculation(X_old %>% select(-all_of(reduce_basic)), "Model bazowy po redukcji")
print(vifs_basic_after_red)

# Tworzenie ostatecznego zbioru cech
X_final <- X_new %>% select(-all_of(reduce_basic))

# Porównanie R-squared
model_new <- lm(Pierścienie ~ ., data = cbind(X_old, Pierścienie = y_local$Pierścienie))
model_final <- lm(Pierścienie ~ ., data = cbind(X_old %>% select(-all_of(reduce_basic)), Pierścienie = y_local$Pierścienie))

cat("Score of initial model =", summary(model_new)$r.squared, "\n")
cat("Score of final model =", summary(model_final)$r.squared, "\n")

# F TEST
n_perm <- 1000  # Zmniejszona liczba permutacji dla demonstracji
f_name <- 'F_test_before_reduction'

F_test_before_reduction <- F_test_permutation(X_final, y_local$Pierścienie, NEW_FEATURE_NAMES, n_perm)
print(F_test_before_reduction)

# KOŃCOWA ANALIZA - VIFY PO REDUKCJI
significant_level <- 0.05
vif_reduce_value <- 12

F_test_before_reduction$`Odrzucenie H0 dla α = 0.05` <- F_test_before_reduction$`P value` < significant_level
reduce_features_F_test <- rownames(F_test_before_reduction)[!F_test_before_reduction$`Odrzucenie H0 dla α = 0.05`]

# Redukcja modelu na podstawie VIF
reduce_features_vif_test <- c()
vifs_after_F_test <- list()
index_name <- "Model rozszerzony przed redukcją vif"

repeat {
  drop_features <- c(reduce_features_F_test, reduce_features_vif_test)
  current_features <- setdiff(colnames(X_final), drop_features)
  local_vifs <- vifs_calculation(X_final %>% select(all_of(current_features)), index_name)
  
  max_vif_column <- names(which.max(local_vifs[1, ]))
  max_vif_value <- max(local_vifs[1, ])
  
  vifs_after_F_test[[length(vifs_after_F_test) + 1]] <- local_vifs
  reduce_features_vif_test <- c(reduce_features_vif_test, max_vif_column)
  
  if (max_vif_value < vif_reduce_value) break
}

vifs_after_F_test <- bind_rows(vifs_after_F_test)

# Porównanie modeli
model_new <- lm(Pierścienie ~ ., data = cbind(X_new, Pierścienie = y_local$Pierścienie))
model_final <- lm(Pierścienie ~ ., data = cbind(X_final, Pierścienie = y_local$Pierścienie))

cat("Score of initial model =", summary(model_new)$r.squared, "\n")
cat("Score of final model =", summary(model_final)$r.squared, "\n")